# Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.sequence import village_bootstrap

# Params

In [ ]:
ARMS = {"control": "033.1", "building": "033.2", "structure": "033.3", "noise": "033.4"}
CONTROL = "control"
LOSS_ARMS = ["building", "structure"]
NOISE_ARM = "noise"
REFERENCE_FLIGHTS = "../027-sequence-filters/artifacts/flights-027.12.csv"
CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                  "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                  "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                  "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]
BAR_RATIO = 0.9
PRECONDITION_DELTA = 0.03
SEG_METRICS = ["building_missed_share_mean", "separable_merge_rate_mean", "track_skeleton_recall_mean",
               "track_skeleton_precision_mean", "miou_mean", "miou_drone-val"]
HEAD_METRICS = ["mean_position_error_m_at_10m", "harm_share_at_10m", "holdout_mean_position_error_m_at_10m",
                "holdout_harm_share_at_10m"]
BOOTSTRAP_RESAMPLES = 10_000
SEED = 42
ARTIFACTS = Path("artifacts")
FIGURES = Path("figures")

# Data

In [ ]:
flights = {arm: pd.read_csv(ARTIFACTS / f"flights-{run}.csv") for arm, run in ARMS.items()
           if (ARTIFACTS / f"flights-{run}.csv").exists()}
flights["027.12"] = pd.read_csv(REFERENCE_FLIGHTS)
{name: len(table) for name, table in flights.items()}

In [ ]:
seg = pd.DataFrame({arm: {k: json.loads((ARTIFACTS / f"segmenter-{arm}" / "metrics.json").read_text())[k]
                         for k in SEG_METRICS} for arm in ARMS}).T
seg

In [ ]:
heads = pd.DataFrame({arm: {k: json.loads((ARTIFACTS / f"head-{arm}.json").read_text())[k] for k in HEAD_METRICS}
                      for arm in ARMS if (ARTIFACTS / f"head-{arm}.json").exists()}).T
heads

# Run

In [ ]:
def precondition(arm: str) -> bool:
    missed_ok = seg.loc[arm, "building_missed_share_mean"] <= seg.loc[CONTROL, "building_missed_share_mean"] - PRECONDITION_DELTA
    if arm == "building":
        return bool(missed_ok)
    recall_ok = seg.loc[arm, "track_skeleton_recall_mean"] >= seg.loc[CONTROL, "track_skeleton_recall_mean"] + PRECONDITION_DELTA
    return bool(missed_ok and recall_ok)

In [ ]:
{arm: precondition(arm) for arm in LOSS_ARMS}

In [ ]:
def system_row(table: pd.DataFrame) -> dict:
    clean = table[table["village"].isin(CLEAN_VILLAGES)]
    return {"flights": len(table), "mean_end_error_m": table["end_error_m"].mean(),
            "median_end_error_m": table["end_error_m"].median(), "divergence_share": table["diverged"].mean(),
            "clean_flights": len(clean), "clean_mean_end_error_m": clean["end_error_m"].mean()}

In [ ]:
system = pd.DataFrame({name: system_row(table) for name, table in flights.items()}).T
system

In [ ]:
def paired(a: str, b: str, villages: list[str] | None = None) -> dict:
    fa, fb = flights[a], flights[b]
    if villages is not None:
        fa, fb = fa[fa["village"].isin(villages)], fb[fb["village"].isin(villages)]
    delta, low, high = village_bootstrap(fa, fb, "end_error_m", BOOTSTRAP_RESAMPLES, SEED)
    return {"a": a, "b": b, "subset": "clean" if villages else "all", "delta_m": delta, "ci_low": low, "ci_high": high}

In [ ]:
pairs = [(arm, CONTROL) for arm in LOSS_ARMS + [NOISE_ARM]] + [(CONTROL, "027.12")]
comparisons = pd.DataFrame([paired(a, b, v) for a, b in pairs if a in flights and b in flights
                            for v in (None, CLEAN_VILLAGES)])
comparisons

In [ ]:
def bar_verdict(arm: str) -> dict:
    control, row = system.loc[CONTROL], system.loc[arm]
    diff = comparisons[(comparisons["a"] == arm) & (comparisons["b"] == CONTROL) & (comparisons["subset"] == "all")].iloc[0]
    noise_gap = abs(system.loc[CONTROL, "mean_end_error_m"] - system.loc[NOISE_ARM, "mean_end_error_m"])
    checks = {"1_ratio": row["mean_end_error_m"] <= BAR_RATIO * control["mean_end_error_m"],
              "1_ci_clear_of_0": diff["ci_high"] < 0,
              "2_beyond_noise": control["mean_end_error_m"] - row["mean_end_error_m"] > noise_gap,
              "3_divergence": row["divergence_share"] <= control["divergence_share"],
              "4_clean_guard": row["clean_mean_end_error_m"] <= control["clean_mean_end_error_m"]}
    return {"arm": arm, "precondition": precondition(arm), **checks, "passes": all(checks.values()),
            "noise_gap_m": noise_gap}

In [ ]:
verdicts = pd.DataFrame([bar_verdict(arm) for arm in LOSS_ARMS if arm in flights and NOISE_ARM in flights])
verdicts

In [ ]:
results = {"segmentation": seg.to_dict("index"), "heads": heads.to_dict("index"), "system": system.to_dict("index"),
           "comparisons": comparisons.to_dict("records"), "verdicts": verdicts.to_dict("records")}
(ARTIFACTS / "analysis.json").write_text(json.dumps(results, indent=1, default=float))

# Plots

In [ ]:
FIGURES.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_end_error_cdf({name: table["end_error_m"] for name, table in flights.items()}, ax)
ax.set_title("033: end error on the test flights, filter A + own head")
fig.tight_layout()
fig.savefig(FIGURES / "end-error-cdf.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, metric in zip(axes, ["building_missed_share_mean", "separable_merge_rate_mean", "track_skeleton_recall_mean"]):
    seg[metric].plot.bar(ax=ax, color="#2a78d6")
    ax.set_title(metric, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "segmentation.png", dpi=150)